# K-Nearest Neighbors (KNN) - Estudo Aplicado

Este notebook implementa o algoritmo **K-Nearest Neighbors (KNN)** do zero e compara os resultados com a implementação do `scikit-learn`. O dataset utilizado contém notas de alunos em diferentes disciplinas do curso de Ciência da Computação da UFRRJ.

**Objetivo:** Prever o desempenho do aluno na disciplina `TN703` (variável-alvo) com base nas suas notas nas demais disciplinas (`IM885`, `IM853`, `TN707`, `TN705`, `TN706`).


## 1. Imports

Bibliotecas utilizadas:
- **pandas / numpy** — manipulação e computação numérica
- **matplotlib** — visualização
- **scikit-learn** — pré-processamento, KNN de referência, métricas e PCA
- **math** — cálculo da raiz quadrada na implementação manual


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, ConfusionMatrixDisplay

from math import sqrt


## 2. Carregamento e Exploração dos Dados

O dataset contém notas (0–10) de 243 alunos em 6 disciplinas. A coluna `matricula` é um identificador e não entra no modelo.


In [ ]:
alunos = pd.read_csv('dataset.csv')
print(f"Shape: {alunos.shape}")
alunos.head()


In [ ]:
# Distribuição estatística das notas brutas
alunos.drop(columns='matricula').describe().round(2)


## 3. Classificação das Notas

Para usar KNN precisamos de rótulos discretos. Nós optamos por definir 2 métodos de classificação:

| Modo | Descrição |
|------|-----------|
| `binary` | 0 = reprovado (nota < 5), 1 = aprovado (nota ≥ 5) |
| `4groups` | 1 = muito baixo (0–2,4), 2 = baixo (2,5–4,9), 3 = médio (5–7,4), 4 = alto (7,5–10) |

> **Configuração:** altere a variável `MODE` abaixo para trocar entre os modos.


In [ ]:
MODE = '4groups'  # Altere para 'binary' se desejar apenas aprovado/reprovado

def make_classifier(mode):
    """Retorna uma função que converte uma nota numérica em rótulo de classe."""
    def classify(grade):
        if mode == 'binary':
            return 1 if grade >= 5 else 0
        elif mode == '4groups':
            if 0.0 <= grade < 2.5:  return 1
            elif 2.5 <= grade < 5.0: return 2
            elif 5.0 <= grade < 7.5: return 3
            elif 7.5 <= grade <= 10: return 4
            else: raise ValueError(f"Nota fora do intervalo [0, 10]: {grade}")
        else:
            raise ValueError(f"Modo inválido: '{mode}'. Use 'binary' ou '4groups'.")
    return classify


## 4. Pré-processamento

### 4.1 Seleção de colunas e aplicação da classificação

Selecionamos as 6 colunas de notas, removemos linhas com valores ausentes e convertemos cada nota em sua respectiva classe.

- **Features (X):** `IM885`, `IM853`, `TN707`, `TN705`, `TN706` — as 5 primeiras colunas
- **Target (y):** `TN703` — a última coluna (nota que queremos prever)


In [ ]:
FEATURES = ['IM885', 'IM853', 'TN707', 'TN705', 'TN706', 'TN703']

alunos_clean = alunos.dropna(subset=FEATURES).copy()
data = alunos_clean[FEATURES].map(make_classifier(MODE))

print(f"Amostras após limpeza: {len(data)}")
data.head()


### 4.2 Distribuição das classes no target

Antes de treinar qualquer modelo, é fundamental entender a distribuição das classes. **Desbalanceamento severo** pode fazer o modelo ignorar as classes minoritárias.


In [ ]:
class_counts = data['TN703'].value_counts().sort_index()
print("Distribuição das classes em TN703 (target):")
print(class_counts)
print()
print(f"Classe 2 representa apenas {class_counts.get(2, 0)/len(data):.1%} do total — extremamente sub-representada.")

class_counts.plot(kind='bar', color=['#e74c3c','#f39c12','#2ecc71','#3498db'],
                  edgecolor='black', figsize=(6,4))
plt.title('Distribuição das Classes no Target (TN703)')
plt.xlabel('Classe')
plt.ylabel('Quantidade de alunos')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 4.3 Divisão Treino/Teste e Padronização

**Divisão:** usamos 50% dos dados para treino e 50% para teste (`test_size=0.5`). Uma divisão 70/30 ou 80/20 seria mais comum, mas mantemos 50/50 para paridade com o estudo original.

**Padronização (StandardScaler):** o KNN é baseado em distâncias — se uma feature tem escala muito maior que outra (ex: notas de 0–10 vs. valores de 0–1000), ela dominará artificialmente o cálculo da distância. `StandardScaler` resolve isso ao transformar cada feature para média 0 e desvio padrão 1.

> ⚠️ O `fit` do scaler é feito **somente no conjunto de treino** para evitar *data leakage* (vazamento de informação do teste para o treino).


In [ ]:
X = data.iloc[:, :-1].values  # features: IM885, IM853, TN707, TN705, TN706
y = data.iloc[:, -1].values   # target: TN703

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=0
)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)   # fit + transform no treino
X_test  = sc.transform(X_test)        # apenas transform no teste (evita data leakage)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")


## 5. Implementação Manual do KNN

Implementamos o KNN do zero para entender o algoritmo por dentro. A lógica central é:

1. **`calculate_euclidean`** - calcula a distância euclidiana entre dois pontos:  
   $d(p, q) = \sqrt{\sum_{i=1}^{n}(p_i - q_i)^2}$

2. **`nearest_neighbors`** - para um ponto de teste, calcula sua distância a todos os pontos de treino e retorna os `k` mais próximos.

3. **`predict`** - para cada ponto de teste, obtém os `k` vizinhos e aplica **votação por maioria** (a classe mais frequente entre os vizinhos vence).


In [ ]:
class KNN:
    def __init__(self, k):
        self.k = k

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train

    def _euclidean(self, a, b):
        """Distância euclidiana entre dois vetores."""
        return sqrt(sum((ai - bi) ** 2 for ai, bi in zip(a, b)))

    def _nearest_neighbors(self, sample):
        """Retorna os rótulos dos k vizinhos mais próximos de `sample`."""
        distances = [
            (self.y_train[i], self._euclidean(self.X_train[i], sample))
            for i in range(len(self.X_train))
        ]
        distances.sort(key=lambda x: x[1])
        return [label for label, _ in distances[:self.k]]

    def predict(self, X):
        """Prediz a classe de cada amostra em X por votação da maioria."""
        predictions = []
        for sample in X:
            neighbors = self._nearest_neighbors(sample)
            majority = max(set(neighbors), key=neighbors.count)
            predictions.append(majority)
        return predictions


In [ ]:
# Treina o modelo manual com k=5
knn_manual = KNN(k=5)
knn_manual.fit(X_train, y_train)
pred_manual = knn_manual.predict(X_test)
print("Modelo manual treinado e predições geradas.")


## 6. Modelo KNN com scikit-learn

O `KNeighborsClassifier` do scikit-learn usa a **distância de Minkowski** com `p=2`, que é equivalente à distância euclidiana, o mesmo critério da nossa implementação manual. Isso permite uma comparação direta entre os dois modelos.


In [ ]:
knn_sklearn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
knn_sklearn.fit(X_train, y_train)
pred_sklearn = knn_sklearn.predict(X_test)
print("Modelo scikit-learn treinado e predições geradas.")


## 7. Avaliação dos Modelos

### 7.1 Acurácia

A **acurácia** mede a proporção de predições corretas sobre o total de amostras de teste.

$$\text{Acurácia} = \frac{\text{Predições corretas}}{\text{Total de amostras}}$$

> ⚠️ Em datasets desbalanceados (como este, onde a classe 2 tem apenas 3 amostras), a acurácia sozinha pode ser enganosa. 
Um modelo que nunca prevê a classe 2, por exemplo, ainda assim terá acurácia alta. Por isso, analisamos também **precision, recall e F1-score por classe**.


In [ ]:
acc_manual  = accuracy_score(y_test, pred_manual)
acc_sklearn = accuracy_score(y_test, pred_sklearn)

print(f"Acurácia — Modelo Manual:      {acc_manual:.4f}  ({acc_manual:.1%})")
print(f"Acurácia — Modelo scikit-learn: {acc_sklearn:.4f}  ({acc_sklearn:.1%})")


### 7.2 Matriz de Confusão

A **matriz de confusão** mostra, para cada classe real, quantas amostras foram corretamente classificadas (diagonal principal) e para quais classes elas foram confundidas (fora da diagonal).

**Como ler:** linha = classe real | coluna = classe predita.  
- Diagonal principal = acertos  
- Fora da diagonal = erros (confusões entre classes)

> ℹ️ Para classificação multiclasse (3 ou 4 classes), **não existe** o conceito único de Verdadeiro Positivo / Falso Negativo aplicado a uma única matriz. Esses termos só fazem sentido na classificação binária, ou quando calculados *por classe* (one-vs-rest). É necessário se atentar a esse ponto, dependendo do `MODE` escolhido no início do algoritmo.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, preds, title in zip(
    axes,
    [pred_sklearn, pred_manual],
    ['scikit-learn KNN', 'KNN Manual']
):
    cm = confusion_matrix(y_test, preds, labels=sorted(np.unique(y)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=sorted(np.unique(y)))
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)

plt.suptitle('Matrizes de Confusão (Conjunto de Teste)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


### 7.3 Relatório Detalhado por Classe

O relatório abaixo mostra **precision**, **recall** e **F1-score** para cada classe individualmente:

| Métrica | Significado |
|---------|-------------|
| **Precision** | De tudo que o modelo previu como classe X, quanto realmente era X? |
| **Recall** | De todas as amostras reais da classe X, quantas o modelo acertou? |
| **F1-score** | Média harmônica entre precision e recall (equilíbrio entre os dois) |


In [ ]:
print("=== scikit-learn KNN ===")
print(classification_report(y_test, pred_sklearn, zero_division=0))

print("=== KNN Manual ===")
print(classification_report(y_test, pred_manual, zero_division=0))


## 8. Visualização das Fronteiras de Decisão (PCA 2D)

Como trabalhamos com 5 features, não é possível plotar as fronteiras de decisão diretamente. Usamos **PCA (Análise de Componentes Principais)** para reduzir as 5 dimensões para 2, permitindo a visualização.

> ⚠️ Para essa visualização, o classificador é retreinado **nos dados projetados pelo PCA**. Isso significa que as métricas do gráfico são ligeiramente diferentes das métricas calculadas na seção anterior (que usam as 5 features originais). O gráfico serve apenas para **ilustração visual** das regiões de decisão


In [ ]:
# Reduz para 2 componentes principais
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

variancia = pca.explained_variance_ratio_
print(f"Variância explicada: PC1={variancia[0]:.2%}, PC2={variancia[1]:.2%}, Total={sum(variancia):.2%}")

# Retreina o classificador no espaço PCA
knn_pca = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
knn_pca.fit(X_train_pca, y_train)
pred_pca = knn_pca.predict(X_test_pca)
print(f"Acurácia no espaço PCA (apenas para visualização): {accuracy_score(y_test, pred_pca):.2%}")


In [ ]:
classes_presentes = sorted(np.unique(y))
palette = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']
colors  = [palette[c - 1] for c in classes_presentes]
cmap_bg = ListedColormap(colors)

step = 0.01
x_min, x_max = X_test_pca[:, 0].min() - 1, X_test_pca[:, 0].max() + 1
y_min, y_max = X_test_pca[:, 1].min() - 1, X_test_pca[:, 1].max() + 1

X1, X2 = np.meshgrid(
    np.arange(x_min, x_max, step),
    np.arange(y_min, y_max, step)
)
Z = knn_pca.predict(np.c_[X1.ravel(), X2.ravel()]).reshape(X1.shape)

plt.figure(figsize=(8, 6))
plt.contourf(X1, X2, Z, alpha=0.35, cmap=cmap_bg)

for i, cls in enumerate(classes_presentes):
    mask = y_test == cls
    plt.scatter(
        X_test_pca[mask, 0], X_test_pca[mask, 1],
        c=colors[i], label=f'Classe {cls}',
        edgecolors='black', linewidths=0.5, s=60
    )

plt.title('Fronteiras de Decisão — KNN (PCA 2D, Conjunto de Teste)', fontsize=12)
plt.xlabel(f'Componente Principal 1 ({variancia[0]:.1%} da variância)')
plt.ylabel(f'Componente Principal 2 ({variancia[1]:.1%} da variância)')
plt.legend()
plt.tight_layout()
plt.show()


## 9. Análise dos Resultados

### 9.1 Desempenho Geral

O modelo atingiu aproximadamente **65–66% de acurácia** no conjunto de teste — um resultado razoável dado o desbalanceamento severo das classes.

### 9.2 O Problema da Classe 2 (nota entre 2,5 e 4,9)

**A classe 2 nunca foi predita.** Identificamos que isso não é erro nos algoritmos, mas sim uma consequência direta do desbalanceamento extremo dos dados da nossa base:

| Classe | Descrição | Amostras no dataset |
|--------|-----------|---------------------|
| 1 | Muito baixo (0–2,4) | 19 (7,8%) |
| **2** | **Baixo (2,5–4,9)** | **3 (1,2%)** |
| 3 | Médio (5–7,4) | 98 (40,3%) |
| 4 | Alto (7,5–10) | 123 (50,6%) |

Com apenas 3 amostras de classe 2 no dataset inteiro, é quase certo que nenhuma (ou nenhuma relevante) entre no conjunto de treino. Mesmo que entre, os 5 vizinhos mais próximos de qualquer ponto de teste quase sempre serão das classes 3 ou 4, que têm ordens de magnitude mais representantes.

**O classificador não teve dados suficientes para aprender a classe 2.**

### 9.3 Comparação: Modelo Manual vs. scikit-learn

Ambos os modelos usam exatamente a mesma lógica (distância euclidiana, k=5, votação por maioria) e produzem resultados praticamente idênticos. A implementação manual serve como prova de conceito, em produção é preferível utilizar a implementação do scikit-learn, por sua robustez e otimização.

### 9.4 Limitações e Próximos Passos

- **Desbalanceamento:** técnicas como SMOTE (sobreamostragem da classe minoritária) ou ajuste de pesos das classes poderiam melhorar o recall da classe 2.
- **Escolha de k:** um estudo de *cross-validation* variando k (ex: 1 a 20) ajudaria a encontrar o valor ótimo.
- **Divisão treino/teste:** uma divisão 80/20 ou 70/30 seria mais convencional e deixaria mais dados para treino.
- **Outras métricas:** em problemas com desbalanceamento, F1-score macro e AUC-ROC são mais informativos que a acurácia isolada.


In [ ]:
# Resumo final das predições de ambos os modelos
print("Distribuição das predições — scikit-learn:")
unique_sk, counts_sk = np.unique(pred_sklearn, return_counts=True)
for cls, cnt in zip(unique_sk, counts_sk):
    print(f"  Classe {cls}: {cnt} predições")

print()
print("Distribuição das predições — Modelo Manual:")
unique_m, counts_m = np.unique(pred_manual, return_counts=True)
for cls, cnt in zip(unique_m, counts_m):
    print(f"  Classe {cls}: {cnt} predições")

print()
print("Distribuição real no conjunto de teste:")
unique_r, counts_r = np.unique(y_test, return_counts=True)
for cls, cnt in zip(unique_r, counts_r):
    print(f"  Classe {cls}: {cnt} amostras reais")
